# VINDATATHON 2026 - GRIDBREAKER

## Part 01: MCQs Multiple Choice Questions

## Overview

This section contains the logic and execution script (`mcq_results.py`) to programmatically solve the 10 Multiple Choice Questions from **Part 1 of Datathon 2026: The Gridbreakers**. 

This script loads the provided CSV files, performs the required data aggregations and joins, and outputs the final answers (A, B, C, or D) to ensure 100% reproducibility.

### 0. Environment Dependencies

In [84]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "../../data/raw/"

### 1. Load dataset

In [95]:
print("Loading datasets...")

df_orders      = pd.read_csv(DATA_DIR + "orders.csv", parse_dates=["order_date"])
df_order_items = pd.read_csv(DATA_DIR + "order_items.csv", low_memory=False)
df_products    = pd.read_csv(DATA_DIR + "products.csv")
df_promotions  = pd.read_csv(DATA_DIR + "promotions.csv", parse_dates=["start_date", "end_date"])
df_customers   = pd.read_csv(DATA_DIR + "customers.csv", parse_dates=["signup_date"])
df_geography   = pd.read_csv(DATA_DIR + "geography.csv")
df_payments    = pd.read_csv(DATA_DIR + "payments.csv")
df_returns     = pd.read_csv(DATA_DIR + "returns.csv", parse_dates=["return_date"])
df_reviews     = pd.read_csv(DATA_DIR + "reviews.csv", parse_dates=["review_date"])
df_shipments   = pd.read_csv(DATA_DIR + "shipments.csv", parse_dates=["ship_date", "delivery_date"])
df_inventory   = pd.read_csv(DATA_DIR + "inventory.csv", parse_dates=["snapshot_date"])
df_web_traffic = pd.read_csv(DATA_DIR + "web_traffic.csv", parse_dates=["date"])
df_sales       = pd.read_csv(DATA_DIR + "sales.csv", parse_dates=["Date"])

print("All datasets loaded.\n")

Loading datasets...
All datasets loaded.



### 2. Answers

#### Question 1

**Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)**

A) 30 ngày

B) 90 ngày

C) 144 ngày

D) 365 ngày

In [111]:
print("="*150)
print("Table orders:")
print("="*150)
display(df_orders.head())
display(df_orders.info())

print("="*150)
df_orders = df_orders.sort_values(['customer_id', 'order_date'])
df_orders['prev_order_date'] = df_orders.groupby('customer_id')['order_date'].shift(1)
df_orders['gap'] = (df_orders['order_date'] - df_orders['prev_order_date']).dt.days
display(df_orders.head())
median_gap = df_orders[df_orders['gap'].notnull()]['gap'].median()
print(f"Inter-Order Gap: {median_gap}")
print("="*150)
print("Choose C.")

# re-raw data
df_orders = df_orders.drop(['prev_order_date', 'gap'], axis=1)

Table orders:


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,gap
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search,NaN
143252,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral,675.0
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,426.0
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,632.0
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,1037.0


<class 'pandas.core.frame.DataFrame'>
Index: 646945 entries, 4023 to 637881
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  object        
 5   payment_method  646945 non-null  object        
 6   device_type     646945 non-null  object        
 7   order_source    646945 non-null  object        
 8   gap             556699 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(4)
memory usage: 49.4+ MB


None

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,gap,prev_order_date
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search,NaN,NaT
143252,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral,675.0,2012-07-25
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,426.0,2014-05-31
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,632.0,2015-07-31
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,1037.0,2017-04-23


Inter-Order Gap: 144.0
Choose C.


#### Question 2

**Q2. Phân khúc sản phẩm (`segment`) nào trong `products.csv` có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức `(price − cogs)/price`?**

A) Premium

B) Performance

C) Activewear

D) Standard

In [117]:
print("="*150)
print("Table products:")
print("="*150)
display(df_products.head())
display(df_products.info())

print("="*150)
print("Formula: profit_margin = (price - cogs)/price")
df_products['profit_margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']
display(df_products.head())
avg_margin_by_segment = (
    df_products
    .groupby('segment', as_index=False)['profit_margin']
    .mean()
    .rename(columns={'profit_margin': 'avg_profit_margin'})
    .sort_values("avg_profit_margin", ascending=False)
)
display(pd.DataFrame(avg_margin_by_segment).style.highlight_max(subset=['avg_profit_margin'], color='lightgreen'))
best_segment = avg_margin_by_segment.loc[
    avg_margin_by_segment['avg_profit_margin'].idxmax(),
    'segment'
]
print(f"Segment with the highest average of profit margin: {best_segment}.")
print("="*150)
print("Choose D.")

# re-raw data
df_products = df_products.drop(['profit_margin'], axis=1)

Table products:


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB


None

Formula: profit_margin = (price - cogs)/price


,product_id,product_name,category,segment,size,color,price,cogs,profit_margin
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278,0.2871
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954,0.4558
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406,0.1080


,segment,avg_profit_margin
6,Standard,0.313442
5,Premium,0.285377
1,All-weather,0.284176
0,Activewear,0.265600
4,Performance,0.263650
2,Balanced,0.258038
7,Trendy,0.240758
3,Everyday,0.236343


Segment with the highest average of profit margin: Standard.
Choose D.


#### Question 3

**Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join `returns` với `products` theo `product_id`), lý do trả hàng nào xuất hiện nhiều nhất?**

A) defective

B) wrong_size

C) changed_mind

D) not_as_described

In [120]:
print("="*150)
print("Table products:")
print("="*150)
display(df_products.head())
display(df_products.info())
print(df_products['category'].value_counts())

print("Table returns:")
print("="*150)
display(df_returns.head())
display(df_returns.info())

print("="*150)
df_q3 = pd.merge(df_returns, df_products, on='product_id')
streetwear_returns = df_q3[df_q3['category'] == 'Streetwear']
streetwear_returns_stats = streetwear_returns['return_reason'].value_counts()
display(pd.DataFrame(streetwear_returns_stats).style.highlight_max(subset=['count'], color='lightgreen'))
most_common_reason = streetwear_returns['return_reason'].mode().iloc[0]
print(f"Most common reason for Streetwear returns: {most_common_reason}")
print("="*150)
print("Choose B.")


Table products:


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB


None

category
Streetwear    1320
Outdoor        743
Casual         201
GenZ           148
Name: count, dtype: int64
Table returns:


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   return_id        39939 non-null  object        
 1   order_id         39939 non-null  int64         
 2   product_id       39939 non-null  int64         
 3   return_date      39939 non-null  datetime64[ns]
 4   return_reason    39939 non-null  object        
 5   return_quantity  39939 non-null  int64         
 6   refund_amount    39939 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 2.1+ MB


None

,count
return_reason,
wrong_size,7626
defective,4330
not_as_described,3854
changed_mind,3830
late_delivery,2159


Most common reason for Streetwear returns: wrong_size
Choose B.


#### Question 4

**Q4. Trong `web_traffic.csv`, nguồn truy cập (`traffic_source`) nào có tỷ lệ thoát trung bình (`bounce_rate`) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột `traffic_source`?**

A) organic_search

B) paid_search

C) email_campaign

D) social_media

In [123]:
print("="*150)
print("Table web traffic:")
print("="*150)
display(df_web_traffic.head())
display(df_web_traffic.info())
print(df_web_traffic['traffic_source'].value_counts())

print("="*150)
source_stats = (
    df_web_traffic
    .groupby('traffic_source')['bounce_rate'].mean()
    .rename('avg_bounce_rate')
    .sort_values(ascending=True)
)
display(pd.DataFrame(source_stats).style.highlight_min(subset=['avg_bounce_rate'], color='lightgreen'))

print(f"Traffic source with the minimum bounce rate: {source_stats.idxmin()}")
print("="*150)
print("Choose C.")


Table web traffic:


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      3652 non-null   datetime64[ns]
 1   sessions                  3652 non-null   int64         
 2   unique_visitors           3652 non-null   int64         
 3   page_views                3652 non-null   int64         
 4   bounce_rate               3652 non-null   float64       
 5   avg_session_duration_sec  3652 non-null   float64       
 6   traffic_source            3652 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(1)
memory usage: 199.8+ KB


None

traffic_source
organic_search    1090
paid_search        784
social_media       632
email_campaign     505
referral           375
direct             266
Name: count, dtype: int64


,avg_bounce_rate
traffic_source,
email_campaign,0.004458
social_media,0.004476
paid_search,0.004478
referral,0.004499
organic_search,0.004504
direct,0.004511


Traffic source with the minimum bounce rate: email_campaign
Choose C.


#### Question 5

**Q5. Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?**

A) 12%

B) 25%

C) 39%

D) 54%

In [16]:
print("="*100)
print("Table order_items:")
print("="*100)
display(df_order_items.head())
display(df_order_items.info())

print("="*100)
total_rows = len(df_order_items)
promo_not_null = (df_order_items["promo_id"].notna() | df_order_items["promo_id_2"].notna()).sum()
pct_promo = promo_not_null / total_rows * 100

print(f"Total order items: {total_rows}")
print(f"Total order items that applied promotion: {promo_not_null}")
print(f"=> Percentage: {pct_promo:.1f}%")
print("="*100)
print("Choose C.")


Table order_items:


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB


None

Total order items: 714669
Total order items that applied promotion: 276316
=> Percentage: 38.7%
Choose C.


#### Question 6

**Q6. Trong customers.csv, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)**

A) 55+

B) 25–34

C) 35–44

D) 45–54

In [42]:
print("="*100)
print("Table customers:")
print("="*100)
display(df_customers.head())
display(df_customers.info())
print(f"Check value of {df_customers['age_group'].value_counts()}\n")

print("="*100)
print("Table orders:")
print("="*100)
display(df_orders.head())
display(df_orders.info())

print("="*100)
not_null_customers = df_customers[df_customers["age_group"].notna()][["customer_id", "age_group"]]
orders_per_cust = df_orders.groupby("customer_id").size().reset_index(name="n_orders")
 
df_q6 = not_null_customers.merge(orders_per_cust, on="customer_id", how="left").fillna(0)
age_stats = (
    df_q6
    .groupby("age_group")
    .agg(
        n_customers=("customer_id", "count"),
        total_orders=("n_orders", "sum"),
    )
    .assign(avg_orders=lambda d: d["total_orders"] / d["n_customers"])
    .sort_values("avg_orders", ascending=False)
)

print("Age group statistic:")
display(age_stats.style.highlight_max(subset=['avg_orders'], color='lightgreen'))
print(f"Age group that has the highest average orders per customer: {age_stats['avg_orders'].idxmax()}")
# display(age_stats.head(1))
print("="*100)
print("Choose A.")


Table customers:


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          121930 non-null  int64         
 1   zip                  121930 non-null  int64         
 2   city                 121930 non-null  object        
 3   signup_date          121930 non-null  datetime64[ns]
 4   gender               121930 non-null  object        
 5   age_group            121930 non-null  object        
 6   acquisition_channel  121930 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 6.5+ MB


None

Check value of age_group
25-34    36342
35-44    31920
45-54    23172
18-24    17039
55+      13457
Name: count, dtype: int64

Table orders:


...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  object        
 5   payment_method  646945 non-null  object        
 6   device_type     646945 non-null  object        
 7   order_source    646945 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 39.5+ MB


None

Age group statistic:


,n_customers,total_orders,avg_orders
age_group,,,
55+,13457,72760.000000,5.406851
45-54,23172,124138.000000,5.357241
35-44,31920,170368.000000,5.337343
25-34,36342,190622.000000,5.245226
18-24,17039,89057.000000,5.226656


Age group that has the highest average orders per customer: 55+
Choose A.


#### Question 7

**Q7. Vùng (`region`) nào trong `geography.csv` tạo ra tổng doanh thu cao nhất trong `sales_train.csv`?**

A) West

B) Central

C) East

D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

In [104]:
print("="*100)
print("Table geography:")
print("="*100)
display(df_geography.head())
display(df_geography.info())
print(df_geography['region'].value_counts(), "\n")

print("="*100)
print("Table sales:")
print("="*100)
display(df_sales.head())

print(
    "There is absolutely no order_id or zip to map to the geographic area. "
    "Therefore, 'Revenue' here must \nactually be calculated from the details "
    "of transactions during the training phase by combining the\n"
    "`orders`, `order_items`, and `geography` tables.\n"
)

print("The time period to follow sales_train is from 04/07/2012 to 31/12/2022.")
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
start_date = pd.to_datetime('2012-07-04')
end_date = pd.to_datetime('2022-12-31')
df_orders_train = df_orders[(df_orders['order_date'] >= start_date) & (df_orders['order_date'] <= end_date)]

print(f"-- Filtered orders between {start_date.date()} and {end_date.date()}.")
print(f"-- Total orders before filter: {len(df_orders)} | After filter: {len(df_orders_train)}\n")

print("="*100)
print("How to calculate the Revenue column in sales.csv?")

print("Gross Demand = Quantity x Unit Price")
print("Net Revenue = Quantity x Unit Price - Discount Amount")

# check revenue mapping
df_order_items["gross_demand"] = (
    df_order_items["quantity"] * df_order_items["unit_price"]
)
df_order_items["net_revenue"] = (
    df_order_items["quantity"] * df_order_items["unit_price"] - df_order_items["discount_amount"]
)
df_trans = pd.merge(
    df_order_items[['order_id', 'gross_demand', 'net_revenue']], 
    df_orders[['order_id', 'order_date', 'order_status']], 
    on='order_id', 
    how='inner'
)
gross_demand = df_trans.groupby('order_date')['gross_demand'].sum().reset_index()
gross_demand.rename(columns={'order_date': 'Date', 'gross_demand': 'Gross_Demand'}, inplace=True)
net_revenue = df_trans.groupby('order_date')['net_revenue'].sum().reset_index()
net_revenue.rename(columns={'order_date': 'Date', 'net_revenue': 'Net_Revenue'}, inplace=True)

df_compare = pd.merge(df_sales[['Date', 'Revenue']], gross_demand, on='Date', how='outer')
df_compare = pd.merge(df_compare, net_revenue, on='Date', how='outer')
display(df_compare.head())
display(df_compare.describe())
df_compare['is_match_a'] = np.isclose(df_compare['Revenue'], df_compare['Gross_Demand'], atol=0.01)
df_compare['is_match_b'] = np.isclose(df_compare['Revenue'], df_compare['Net_Revenue'], atol=0.01)
print(
    f"Rows are not matched:\n"
    f"   Gross Demand - {len(df_compare[~df_compare['is_match_a']])},\n"
    f"   Net Revenue  - {len(df_compare[~df_compare['is_match_b']])}.\n"
)
print("Conclusion: Revenue in sales.csv = quantity x unit_price = gross_demand")

print("="*100)
order_rev = df_order_items.groupby("order_id")["gross_demand"].sum().reset_index()
 
# Join orders with geography
order_geo = (
    df_orders_train[["order_id", "zip"]]
    .merge(df_geography[["zip", "region"]], on="zip", how="left")
)
 
region_revenue = (
    order_geo
    .merge(order_rev, on="order_id", how="left")
    .groupby("region")["gross_demand"]
    .sum()
    .sort_values(ascending=False)
)

print("\nRevenue by region:")
# display(region_revenue)
display(
    pd.DataFrame(region_revenue).style
    .format({'gross_demand': '${:,.2f}'})
    .highlight_max(subset=['gross_demand'], color='lightgreen')
)
print(f"The highest total sales region: {region_revenue.idxmax()}")
print("="*100)
print("Choose A.")

# re-raw data
df_order_items = df_order_items.drop(['gross_demand', 'net_revenue'], axis=1)
#df_order_items.columns

Table geography:


,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB


None

region
East       18929
Central    14512
West        6507
Name: count, dtype: int64 

Table sales:


...

,Date,Revenue,Gross_Demand,Net_Revenue
0,2012-07-04,5123547.94,5123547.94,5123547.94
1,2012-07-05,2751773.45,2751773.45,2751773.45
2,2012-07-06,3054029.42,3054029.42,3054029.42
3,2012-07-07,2667930.94,2667930.94,2667930.94
4,2012-07-08,2360851.90,2360851.90,2360851.90


,Date,Revenue,Gross_Demand,Net_Revenue
count,3833,3.833000e+03,3.833000e+03,3.833000e+03
mean,2017-10-02 00:00:00,4.286584e+06,4.286584e+06,4.091017e+06
min,2012-07-04 00:00:00,2.798139e+05,2.798139e+05,2.798139e+05
25%,2015-02-17 00:00:00,2.471089e+06,2.471089e+06,2.337998e+06
50%,2017-10-02 00:00:00,3.647304e+06,3.647304e+06,3.489546e+06
75%,2020-05-17 00:00:00,5.350877e+06,5.350877e+06,5.108046e+06
max,2022-12-31 00:00:00,2.090527e+07,2.090527e+07,2.090527e+07
std,NaN,2.624840e+06,2.624840e+06,2.568281e+06


Rows are not matched:
   Gross Demand - 0,
   Net Revenue  - 1707.

Conclusion: Revenue in sales.csv = quantity x unit_price = gross_demand

Revenue by region:


,gross_demand
region,
East,"$7,637,532,676.20"
Central,"$4,941,908,471.68"
West,"$3,851,035,437.65"


The highest total sales region: East
Choose A.


#### Question 8

**Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?**

A) credit_card

B) cod

C) paypal

D) bank_transfer

In [57]:
print("="*100)
print("Table orders:")
print("="*100)
display(df_orders.head())
display(df_orders.info())
print(df_orders['order_status'].value_counts(), "\n")
print(df_orders['payment_method'].value_counts(), "\n")

print("="*100)
cancelled = df_orders[df_orders["order_status"] == "cancelled"]
pay_counts = cancelled["payment_method"].value_counts()
#display(pay_counts)
total_cancelled = len(cancelled)

print("Payment method statistic:")
# display(region_revenue)
display(
    pd.DataFrame(pay_counts).style
    .highlight_max(subset=['count'], color='lightgreen')
)
print(f"The most popular payment method: {pay_counts.idxmax()}")
print("="*100)
print("Choose A.")

Table orders:


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  object        
 5   payment_method  646945 non-null  object        
 6   device_type     646945 non-null  object        
 7   order_source    646945 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 39.5+ MB


None

order_status
delivered    516716
cancelled     59462
returned      36142
shipped       13773
paid          13577
created        7275
Name: count, dtype: int64 

payment_method
credit_card      356352
paypal            97018
cod               96681
apple_pay         64763
bank_transfer     32131
Name: count, dtype: int64 

Payment method statistic:


,count
payment_method,
credit_card,28452
cod,15468
paypal,7817
apple_pay,5190
bank_transfer,2535


The most popular payment method: credit_card
Choose A.


#### Question 9

**Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong `returns` chia cho số dòng trong `order_items` (join với `products` theo `product_id`)?**

A) S

B) M

C) L

D) XL

In [133]:
print("="*100)
print("Table returns:")
print("="*100)
display(df_returns.head())
display(df_returns.info())

print("="*100)
print("Table products:")
print("="*100)
display(df_products.head())
display(df_products.info())
print(df_products['size'].value_counts(), "\n")

print("="*100)
print("Table order_items:")
print("="*100)
display(df_order_items.head())
display(df_order_items.info())

print("="*100)
order_items_size = (
    df_order_items
    .merge(df_products[["product_id", "size"]], on="product_id", how="left")
    .groupby("size")
    .size()
    .rename("n_order_items")
)
returns_size = (
    df_returns
    .merge(df_products[["product_id", "size"]], on="product_id", how="left")
    .groupby("size")
    .size()
    .rename("n_returns")
)
return_size_rate = pd.concat([order_items_size, returns_size], axis=1).fillna(0)
return_size_rate["return_rate"] = return_size_rate["n_returns"] / return_size_rate["n_order_items"]
return_size_rate = return_size_rate.sort_values("return_rate", ascending=False)
display(return_size_rate.style.highlight_max(subset=['return_rate'], color='lightgreen'))
print(f"Size with the highest return rate: size {return_size_rate['return_rate'].idxmax()}.")
print("="*100)
print("Choose A.")

Table returns:


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   return_id        39939 non-null  object        
 1   order_id         39939 non-null  int64         
 2   product_id       39939 non-null  int64         
 3   return_date      39939 non-null  datetime64[ns]
 4   return_reason    39939 non-null  object        
 5   return_quantity  39939 non-null  int64         
 6   refund_amount    39939 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 2.1+ MB


None

Table products:


...

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB


None

,n_order_items,n_returns,return_rate
size,,,
S,172042,9723,0.056515
L,173174,9741,0.056250
M,176428,9820,0.055660
XL,193025,10655,0.055200


Size with the highest return rate: size S.
Choose A.


#### Question 10

**Q10. Trong `payments.csv`, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?**

A) 1 kỳ (trả một lần)

B) 3 kỳ

C) 6 kỳ

D) 12 kỳ

In [136]:
print("="*100)
print("Table payments:")
print("="*100)
display(df_payments.head())
display(df_payments.info())

print("="*100)
install_stats = (
    df_payments
    .groupby("installments")["payment_value"]
    .agg(avg_payment="mean", n_count="count", median_payment="median")
    .sort_values("avg_payment", ascending=False)
)
display(install_stats.style.highlight_max(subset=['avg_payment'], color='lightgreen'))
print(f"Installments with the highest average of payment value: {install_stats['avg_payment'].idxmax()}.")
print("="*100)
print("Choose C.")

Table payments:


,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  object 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 19.7+ MB


None

,avg_payment,n_count,median_payment
installments,,,
6,24446.654403,109910,17451.965000
3,24399.635486,218949,17366.720000
12,24245.772694,54126,17336.750000
1,24113.274166,262866,17088.885000
2,708.473729,1094,722.300000


Installments with the highest average of payment value: 6.
Choose C.


## End